In [1]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import math
import warnings

warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('elvisdenverco2024obweekday_export_odbc.csv')
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS
0,11423,2024-08-14 14:09:00,81,en,2024-08-14 13:58:35,2024-08-14 14:09:00,70.113.126.123,https://denver-co-od.etc-research.com/,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11424,2024-08-14 14:09:26,81,en,2024-08-14 14:09:03,2024-08-14 14:09:26,70.113.126.123,https://denver-co-od.etc-research.com/index.ph...,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11425,2024-08-14 14:24:11,81,en,2024-08-14 14:09:28,2024-08-14 14:24:11,70.113.126.123,https://denver-co-od.etc-research.com/index.ph...,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11427,2024-08-14 14:57:00,81,en,2024-08-14 14:36:36,2024-08-14 14:57:00,50.202.171.186,https://denver-co-od.etc-research.com/,-21600,999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11634,2024-08-19 06:18:09,81,en,2024-08-19 06:10:17,2024-08-27 01:02:36,172.58.57.226,https://denver-co-od.etc-research.com/,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df=df[(df['INTERV_INIT']!='999')]

In [4]:
df.head()

,id,Completed,Last_page,Start_language,Date_started,Date_last_action,IP_address,Referring_URL,TIME_ADJUST,INTERV_INIT,...,ELVIS_USER_CHANGE_7_C_REASON,INVITE_EMAIL,INVITE_SMS,INVITE_CALL,INVITE_TOKEN,INVITE_STATUS,EXP_HOMELESSCode,EXP_HOMELESS,_REVERSE_TRIP,GENERATABLE_TRIPS
4,11634,2024-08-19 06:18:09,81,en,2024-08-19 06:10:17,2024-08-27 01:02:36,172.58.57.226,https://denver-co-od.etc-research.com/,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,11635,2024-08-19 06:23:49,81,en,2024-08-19 06:18:11,2024-08-19 06:23:49,172.58.57.226,https://denver-co-od.etc-research.com/index.ph...,-21600,ANC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,11636,2024-08-19 06:28:49,81,en,2024-08-19 06:23:51,2024-08-27 01:04:04,172.58.57.226,https://denver-co-od.etc-research.com/index.ph...,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,11639,2024-08-19 07:10:38,81,en,2024-08-19 06:57:23,2024-08-27 01:04:56,172.59.229.70,https://denver-co-od.etc-research.com/,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,11640,2024-08-19 07:15:09,81,en,2024-08-19 07:10:41,2024-08-27 01:06:12,172.59.229.70,https://denver-co-od.etc-research.com/index.ph...,-21600,ANC,...,\n\n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
def check_all_characters_present(df, columns_to_check):
    # Function to clean a string by removing underscores and square brackets and converting to lowercase
    def clean_string(s):
        return s.replace('_', '').replace('[', '').replace(']', '').replace(' ','').replace('#','').lower()

    # Clean and convert all column names in df to lowercase for case-insensitive comparison
    df_columns_lower = [clean_string(column) for column in df.columns]

    # Clean and convert the columns_to_check list to lowercase for case-insensitive comparison
    columns_to_check_lower = [clean_string(column) for column in columns_to_check]

    # Use a list comprehension to filter columns
    matching_columns = [column for column in df.columns if clean_string(column) in columns_to_check_lower]

    return matching_columns

In [6]:
timestamp_columns_check=['completed','reviewscrtime','homeaddtime']
timestamp_columns=check_all_characters_present(df,timestamp_columns_check)
timestamp_columns.sort()
timestamp_columns

['Completed', 'HOMEADD_TIME', 'REVIEWSCR_TIME']

In [7]:
df[timestamp_columns].info()

<class 'pandas.core.frame.DataFrame'>
Index: 3813 entries, 4 to 3953
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Completed       3813 non-null   object
 1   HOMEADD_TIME    3813 non-null   object
 2   REVIEWSCR_TIME  3813 non-null   object
dtypes: object(3)
memory usage: 119.2+ KB


In [8]:
df[timestamp_columns[0]]=pd.to_datetime(df[timestamp_columns[0]], errors='coerce')
df[timestamp_columns[1]]=pd.to_datetime(df[timestamp_columns[1]], errors='coerce')
df[timestamp_columns[2]]=pd.to_datetime(df[timestamp_columns[2]], errors='coerce')
df[timestamp_columns].info()

<class 'pandas.core.frame.DataFrame'>
Index: 3813 entries, 4 to 3953
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Completed       3813 non-null   datetime64[ns]
 1   HOMEADD_TIME    3813 non-null   datetime64[ns]
 2   REVIEWSCR_TIME  3813 non-null   datetime64[ns]
dtypes: datetime64[ns](3)
memory usage: 119.2 KB


In [9]:
timestamp_columns

['Completed', 'HOMEADD_TIME', 'REVIEWSCR_TIME']

In [10]:
# for _,row in df.iterrows():
#     df.loc[row.name,'Full Survey Time']=''
# #     Logistics Time=HOMEADD_TIME-Completed
#     df.loc[row.name,'Logistics Time']=(row[timestamp_columns[1]]-row[timestamp_columns[0]]).dt.total_seconds() / 60
# #     Demographic Time=HOMEADD_TIME-REVIEWSCR_TIME
#     df.loc[row.name,'Demographic Time']=(row[timestamp_columns[1]]-row[timestamp_columns[0]]).dt.total_seconds() / 60

In [11]:
df[timestamp_columns]

,Completed,HOMEADD_TIME,REVIEWSCR_TIME
4,2024-08-19 06:18:09,2024-08-19 06:13:32,2024-08-19 06:15:09
5,2024-08-19 06:23:49,2024-08-19 06:19:45,2024-08-19 06:20:40
6,2024-08-19 06:28:49,2024-08-19 06:25:33,2024-08-19 06:26:17
7,2024-08-19 07:10:38,2024-08-19 07:07:15,2024-08-19 07:08:30
8,2024-08-19 07:15:09,2024-08-19 07:11:25,2024-08-19 07:12:49
...,...,...,...
3949,2024-08-30 08:44:34,2024-08-30 08:32:39,2024-08-30 08:33:22
3950,2024-08-30 08:49:12,2024-08-30 08:46:06,2024-08-30 08:47:57
3951,2024-08-30 08:53:23,2024-08-30 08:49:55,2024-08-30 08:50:56
3952,2024-08-30 09:03:11,2024-08-30 09:01:27,2024-08-30 09:02:09


In [12]:
df[timestamp_columns[0]][:1]

4   2024-08-19 06:18:09
Name: Completed, dtype: datetime64[ns]

In [13]:
df['Full Survey Time'] = ''
df['Field Reviewed'] = ''
# Calculate minutes for Logistics Time
df['Logistics Time'] = (df[timestamp_columns[0]]- df[timestamp_columns[1]]).dt.total_seconds() / 60
# Calculate minutes for Demographic Time
df['Demographic Time'] = ( df[timestamp_columns[2]] - df[timestamp_columns[1]]).dt.total_seconds() / 60
df['Survey Date'] = df['Completed'].dt.date

In [14]:
timestamp_df=df[['id','INTERV_INIT','Survey Date','Full Survey Time','Logistics Time','Demographic Time','Field Reviewed']]

In [15]:
timestamp_df

,id,INTERV_INIT,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed
4,11634,ANC,2024-08-19,,4.616667,1.616667,
5,11635,ANC,2024-08-19,,4.066667,0.916667,
6,11636,ANC,2024-08-19,,3.266667,0.733333,
7,11639,ANC,2024-08-19,,3.383333,1.250000,
8,11640,ANC,2024-08-19,,3.733333,1.400000,
...,...,...,...,...,...,...,...
3949,18235,JTC,2024-08-30,,11.916667,0.716667,
3950,18236,JTC,2024-08-30,,3.100000,1.850000,
3951,18237,JTC,2024-08-30,,3.466667,1.016667,
3952,18238,JTC,2024-08-30,,1.733333,0.700000,


In [16]:
timestamp_df['Full Survey Time']=timestamp_df['Logistics Time']+timestamp_df['Demographic Time']
timestamp_df.head()

,id,INTERV_INIT,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed
4,11634,ANC,2024-08-19,6.233333,4.616667,1.616667,
5,11635,ANC,2024-08-19,4.983333,4.066667,0.916667,
6,11636,ANC,2024-08-19,4.000000,3.266667,0.733333,
7,11639,ANC,2024-08-19,4.633333,3.383333,1.250000,
8,11640,ANC,2024-08-19,5.133333,3.733333,1.400000,


In [17]:
timestamp_df['Full Survey Time Flag'] = (timestamp_df['Full Survey Time'] < 4.5).astype(int)
timestamp_df['Logistics Time Flag'] = (timestamp_df['Logistics Time'] < 2.5).astype(int)
timestamp_df['Demographic Time Flag'] = (timestamp_df['Demographic Time'] < 2.5).astype(int)
timestamp_df.head()

,id,INTERV_INIT,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed,Full Survey Time Flag,Logistics Time Flag,Demographic Time Flag
4,11634,ANC,2024-08-19,6.233333,4.616667,1.616667,,0,0,1
5,11635,ANC,2024-08-19,4.983333,4.066667,0.916667,,0,0,1
6,11636,ANC,2024-08-19,4.000000,3.266667,0.733333,,1,0,1
7,11639,ANC,2024-08-19,4.633333,3.383333,1.250000,,0,0,1
8,11640,ANC,2024-08-19,5.133333,3.733333,1.400000,,0,0,1


In [18]:
# for _,row in timestamp_df.iterrows():
#     if row['Full Survey Time']<5:
#         timestamp_df.loc[row.name,'Full Survey Time Flag']=1
#     else:
#         timestamp_df.loc[row.name,'Full Survey Time Flag']=0
#     if row['Logistics Time']<2.5:
#         timestamp_df.loc[row.name,'Logistics Time Flag']=1
#     else:
#         timestamp_df.loc[row.name,'Logistics Time Flag']=0
#     if row['Demographic Time']<2.5:
#         timestamp_df.loc[row.name,'Demographic Time Flag']=1
#     else:
#         timestamp_df.loc[row.name,'Demographic Time Flag']=0

In [19]:
timestamp_df[timestamp_df['Demographic Time']>timestamp_df['Logistics Time']]

,id,INTERV_INIT,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed,Full Survey Time Flag,Logistics Time Flag,Demographic Time Flag


In [20]:
# timestamp_df[timestamp_df['INTERV_INIT']=='ANC'].to_csv('Interviewer_TimeStamp_Data.csv',index=False)

In [21]:
timestamp_df.head()

,id,INTERV_INIT,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed,Full Survey Time Flag,Logistics Time Flag,Demographic Time Flag
4,11634,ANC,2024-08-19,6.233333,4.616667,1.616667,,0,0,1
5,11635,ANC,2024-08-19,4.983333,4.066667,0.916667,,0,0,1
6,11636,ANC,2024-08-19,4.000000,3.266667,0.733333,,1,0,1
7,11639,ANC,2024-08-19,4.633333,3.383333,1.250000,,0,0,1
8,11640,ANC,2024-08-19,5.133333,3.733333,1.400000,,0,0,1


In [22]:
grouped_df = timestamp_df.groupby('Survey Date')['id'].apply(list).reset_index()

# Alternatively, to join IDs as a concatenated string
# grouped_df = timestamp_df.groupby('Survey Date')['id'].apply(lambda x: ', '.join(x.astype(str))).reset_index()

# Rename the columns for clarity
grouped_df.columns = ['Survey Date' ,'IDs']

grouped_df


,Survey Date,IDs
0,2024-08-19,"[11634, 11635, 11636, 11639, 11640, 11641, 116..."
1,2024-08-20,"[11768, 11769, 11770, 11771, 11775, 11776, 117..."
2,2024-08-21,"[12326, 12328, 12330, 12331, 12333, 12334, 123..."
3,2024-08-22,"[13440, 13441, 13442, 13444, 13446, 13447, 134..."
4,2024-08-23,"[14564, 14565, 14566, 14567, 14568, 14569, 145..."
5,2024-08-26,"[14639, 14640, 14641, 14642, 14644, 14645, 146..."
6,2024-08-27,"[15647, 15648, 15649, 15650, 15651, 15654, 156..."
7,2024-08-28,"[16502, 16504, 16505, 16506, 16507, 16508, 165..."
8,2024-08-29,"[17378, 17379, 17380, 17381, 17382, 17383, 173..."
9,2024-08-30,"[18233, 18234, 18235, 18236, 18237, 18238, 18239]"


In [23]:
grouped = timestamp_df.groupby('Survey Date')
result_data = []
for date, group in grouped:
    # Filter IDs where Logistics Time is greater than Demographic Time
    logistic_greater = group[group['Logistics Time'] > group['Demographic Time']]
    logistic_ids = logistic_greater['id'].tolist()
    logistic_count = len(logistic_ids)
    
    # Filter IDs where Demographic Time is greater than Logistics Time
    demographic_greater = group[group['Demographic Time'] > group['Logistics Time']]
    demographic_ids = demographic_greater['id'].tolist()
    demographic_count = len(demographic_ids)
    
    # Prepare rows for the new DataFrame
    if logistic_count > 0:
        result_data.append({
            'Survey Date': date,
            'IDs': ', '.join(map(str, logistic_ids)),
            'Full Survey Time': '',
            'Logistic_Time': logistic_count,
            'Demographic_Time': ''
        })
    
    if demographic_count > 0:
        result_data.append({
            'Survey Date': date,
            'IDs': ', '.join(map(str, demographic_ids)),
            'Full Survey Time': '',
            'Logistic_Time': '',
            'Demographic_Time': demographic_count
        })

# Convert the result list to a new DataFrame
result_df = pd.DataFrame(result_data)


In [24]:
result_df

,Survey Date,IDs,Full Survey Time,Logistic_Time,Demographic_Time
0,2024-08-19,"11634, 11635, 11636, 11639, 11640, 11641, 1164...",,60,
1,2024-08-20,"11768, 11769, 11770, 11771, 11775, 11776, 1177...",,66,
2,2024-08-21,"12326, 12328, 12330, 12331, 12333, 12334, 1233...",,443,
3,2024-08-22,"13440, 13441, 13442, 13444, 13446, 13447, 1345...",,470,
4,2024-08-23,"14564, 14565, 14566, 14567, 14568, 14569, 1457...",,65,
5,2024-08-26,"14641, 14642, 14644, 14646, 14647, 14648, 1465...",,489,
6,2024-08-27,"15647, 15648, 15649, 15650, 15651, 15655, 1565...",,449,
7,2024-08-28,"16502, 16504, 16509, 16512, 16517, 16523, 1652...",,382,
8,2024-08-29,"17382, 17383, 17384, 17388, 17395, 17398, 1740...",,480,
9,2024-08-30,"18233, 18234, 18235, 18236, 18237, 18238, 18239",,7,


In [25]:
test_df=df[df['id']==11634][timestamp_columns]
test_df

,Completed,HOMEADD_TIME,REVIEWSCR_TIME
4,2024-08-19 06:18:09,2024-08-19 06:13:32,2024-08-19 06:15:09


In [26]:
logictic_time=(test_df[timestamp_columns[1]] - test_df[timestamp_columns[0]]).dt.total_seconds() / 60
logictic_time

4   -4.616667
dtype: float64

In [27]:
demographic_time=(test_df[timestamp_columns[2]]-test_df[timestamp_columns[1]] ).dt.total_seconds() / 60
demographic_time

4    1.616667
dtype: float64

In [28]:
ts1 = pd.Timestamp('2024-08-14 14:09:00')
ts2 = pd.Timestamp('2024-08-14 14:14:00')
abs(ts1-ts2).total_seconds()/60

5.0

In [29]:
df[df['INTERV_INIT']=='ANC'][['id','Survey Date','Full Survey Time','Logistics Time','Demographic Time','Field Reviewed']]

,id,Survey Date,Full Survey Time,Logistics Time,Demographic Time,Field Reviewed
4,11634,2024-08-19,,4.616667,1.616667,
5,11635,2024-08-19,,4.066667,0.916667,
6,11636,2024-08-19,,3.266667,0.733333,
7,11639,2024-08-19,,3.383333,1.250000,
8,11640,2024-08-19,,3.733333,1.400000,
...,...,...,...,...,...,...
3544,17679,2024-08-29,,2.383333,0.466667,
3556,17693,2024-08-29,,1.733333,0.333333,
3558,17695,2024-08-29,,2.433333,0.933333,
3572,17712,2024-08-29,,3.333333,1.266667,


In [30]:
interviewer_loc_columns_check=['elvisuserloc1lat','elvisuserloc1long','elvisuserloc2lat','elvisuserloc2long','elvisuserloc3lat','elvisuserloc3long','elvisuserloc4lat','elvisuserloc4long','elvisuserloc5lat','elvisuserloc5long','elvisuserloc6lat','elvisuserloc6long']
interviewer_loc_columns=check_all_characters_present(df,interviewer_loc_columns_check)
interviewer_loc_columns

['ELVIS_USER_LOC1_LAT',
 'ELVIS_USER_LOC1_LONG',
 'ELVIS_USER_LOC2_LAT',
 'ELVIS_USER_LOC2_LONG',
 'ELVIS_USER_LOC3_LAT',
 'ELVIS_USER_LOC3_LONG',
 'ELVIS_USER_LOC4_LAT',
 'ELVIS_USER_LOC4_LONG',
 'ELVIS_USER_LOC5_LAT',
 'ELVIS_USER_LOC5_LONG',
 'ELVIS_USER_LOC6_LAT',
 'ELVIS_USER_LOC6_LONG']

In [31]:
# # Function to get the first non-zero pair of latitude and longitude
# def get_first_valid_lat_long(row):
#     for i in range(1, 7):  # Loop through LOC1 to LOC6
#         lat = row[f'ELVIS_USER_LOC{i}_LAT']
#         long = row[f'ELVIS_USER_LOC{i}_LONG']
        
#         # Debugging prints to check values
#         print(f"Checking: LOC{i} LAT = {lat}, LONG = {long}")
        
#         if pd.notna(lat) and pd.notna(long) and lat != 0 and long != 0:  # Check for non-NaN and non-zero values
#             print(f"Valid pair found at LOC{i}: LAT = {lat}, LONG = {long}")
#             return lat, long
    
#     # If no valid pair is found, return NaN
#     print("No valid pair found")
#     return np.nan, np.nan

# # Apply the function to each row
# df['ELVIS_USER_LOC_LAT'], df['ELVIS_USER_LOC_LONG'] = zip(*df.apply(get_first_non_zero_lat_long, axis=1))

In [32]:
for _,row in df.iterrows():
    for i in range(1,7):
        
        if not pd.isna(row[f'ELVIS_USER_LOC{i}_LAT']) and not pd.isna(row[f'ELVIS_USER_LOC{i}_LONG']):
            df.loc[row.name,'ELVIS_USER_LOC_LAT']=row[f'ELVIS_USER_LOC{i}_LAT']
            df.loc[row.name,'ELVIS_USER_LOC_LONG']=row[f'ELVIS_USER_LOC{i}_LONG']
            break
        else:
            df.loc[row.name,'ELVIS_USER_LOC_LAT']=np.nan
            df.loc[row.name,'ELVIS_USER_LOC_LONG']=np.nan

In [33]:
df[['id',*interviewer_loc_columns,'ELVIS_USER_LOC_LAT','ELVIS_USER_LOC_LONG']].tail(10)

,id,ELVIS_USER_LOC1_LAT,ELVIS_USER_LOC1_LONG,ELVIS_USER_LOC2_LAT,ELVIS_USER_LOC2_LONG,ELVIS_USER_LOC3_LAT,ELVIS_USER_LOC3_LONG,ELVIS_USER_LOC4_LAT,ELVIS_USER_LOC4_LONG,ELVIS_USER_LOC5_LAT,ELVIS_USER_LOC5_LONG,ELVIS_USER_LOC6_LAT,ELVIS_USER_LOC6_LONG,ELVIS_USER_LOC_LAT,ELVIS_USER_LOC_LONG
3944,18229,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,NaN,NaN,39.787032,-104.895952
3945,18230,39.767540,-104.882412,39.768551,-104.891642,39.770704,-104.891485,39.768594,-104.891518,39.771637,-104.891697,NaN,NaN,39.767540,-104.882412
3946,18232,39.774269,-104.942705,39.772013,-104.969774,39.775779,-104.955680,39.772638,-104.957602,39.771786,-104.959446,NaN,NaN,39.774269,-104.942705
3947,18233,40.032775,-105.258338,40.036744,-105.259852,40.032750,-105.258448,40.035501,-105.259894,40.035500,-105.259893,NaN,NaN,40.032775,-105.258338
3948,18234,40.037628,-105.254307,40.035663,-105.253308,40.038476,-105.254316,40.036108,-105.253540,40.036095,-105.253519,NaN,NaN,40.037628,-105.254307
3949,18235,40.032268,-105.254032,40.031329,-105.253699,40.031329,-105.253699,40.031329,-105.253699,40.031329,-105.253699,NaN,NaN,40.032268,-105.254032
3950,18236,40.017965,-105.258237,40.028478,-105.258853,40.020318,-105.258780,40.025719,-105.257958,40.025925,-105.258013,NaN,NaN,40.017965,-105.258237
3951,18237,40.034610,-105.258277,40.037226,-105.258596,40.036001,-105.258399,40.036687,-105.258648,40.036809,-105.258576,NaN,NaN,40.034610,-105.258277
3952,18238,NaN,NaN,NaN,NaN,NaN,NaN,40.115845,-105.163701,40.115842,-105.163710,NaN,NaN,40.115845,-105.163701
3953,18239,40.152125,-105.109556,NaN,NaN,40.153390,-105.109151,40.152592,-105.107251,40.152250,-105.107348,NaN,NaN,40.152125,-105.109556


In [34]:
df[['id',*interviewer_loc_columns,'ELVIS_USER_LOC_LAT','ELVIS_USER_LOC_LONG']].tail(10)

,id,ELVIS_USER_LOC1_LAT,ELVIS_USER_LOC1_LONG,ELVIS_USER_LOC2_LAT,ELVIS_USER_LOC2_LONG,ELVIS_USER_LOC3_LAT,ELVIS_USER_LOC3_LONG,ELVIS_USER_LOC4_LAT,ELVIS_USER_LOC4_LONG,ELVIS_USER_LOC5_LAT,ELVIS_USER_LOC5_LONG,ELVIS_USER_LOC6_LAT,ELVIS_USER_LOC6_LONG,ELVIS_USER_LOC_LAT,ELVIS_USER_LOC_LONG
3944,18229,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,39.787032,-104.895952,NaN,NaN,39.787032,-104.895952
3945,18230,39.767540,-104.882412,39.768551,-104.891642,39.770704,-104.891485,39.768594,-104.891518,39.771637,-104.891697,NaN,NaN,39.767540,-104.882412
3946,18232,39.774269,-104.942705,39.772013,-104.969774,39.775779,-104.955680,39.772638,-104.957602,39.771786,-104.959446,NaN,NaN,39.774269,-104.942705
3947,18233,40.032775,-105.258338,40.036744,-105.259852,40.032750,-105.258448,40.035501,-105.259894,40.035500,-105.259893,NaN,NaN,40.032775,-105.258338
3948,18234,40.037628,-105.254307,40.035663,-105.253308,40.038476,-105.254316,40.036108,-105.253540,40.036095,-105.253519,NaN,NaN,40.037628,-105.254307
3949,18235,40.032268,-105.254032,40.031329,-105.253699,40.031329,-105.253699,40.031329,-105.253699,40.031329,-105.253699,NaN,NaN,40.032268,-105.254032
3950,18236,40.017965,-105.258237,40.028478,-105.258853,40.020318,-105.258780,40.025719,-105.257958,40.025925,-105.258013,NaN,NaN,40.017965,-105.258237
3951,18237,40.034610,-105.258277,40.037226,-105.258596,40.036001,-105.258399,40.036687,-105.258648,40.036809,-105.258576,NaN,NaN,40.034610,-105.258277
3952,18238,NaN,NaN,NaN,NaN,NaN,NaN,40.115845,-105.163701,40.115842,-105.163710,NaN,NaN,40.115845,-105.163701
3953,18239,40.152125,-105.109556,NaN,NaN,40.153390,-105.109151,40.152592,-105.107251,40.152250,-105.107348,NaN,NaN,40.152125,-105.109556


In [35]:
df[df['id']==13940][[*interviewer_loc_columns,'ELVIS_USER_LOC_LAT','ELVIS_USER_LOC_LONG']].values

array([[nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan]])

In [36]:
df[df['id']==18238][[*interviewer_loc_columns,'ELVIS_USER_LOC_LAT','ELVIS_USER_LOC_LONG']].values

array([[         nan,          nan,          nan,          nan,
                 nan,          nan,   40.1158445, -105.1637008,
          40.115842 , -105.1637097,          nan,          nan,
          40.1158445, -105.1637008]])

In [37]:
unique_interviewers=df['INTERV_INIT'].unique()

In [38]:
unique_interviewers

array(['ANC', 'LSS', 'KSB', 'CBB', 'MAO', 'MXG', 'JMV', 'DHC', 'LXF',
       'NSH', 'CLB', 'GXG', 'TBT', 'SJG', 'JLD', 'LXO', 'BAM', 'DRM',
       'HJT', 'MAD', 'RLC', 'EJP', 'RAB', 'LDP', 'JQT', 'LAJ', 'DKG',
       'PLD', 'DAF', 'BBM', 'DMD', 'HSD', 'PCS', 'RJG', 'CAB', 'RWZ',
       'JLS', 'JTC', 'Pld'], dtype=object)

In [39]:
# for name in unique_interviewers:
    
#     for _,row in df.iterrows():
#         row['INTERV']

In [40]:
# # Sample DataFrame
# data = {
#     'INTERV_INIT': ['ANC', 'ANC', 'ANC', 'BNC', 'ANC', 'ANC'],
#     'ELVIS_USER_LOC2_LAT': [47.6062, 47.6097, 47.6115, 47.6158, 47.6173, 47.6205],
#     'ELVIS_USER_LOC2_LONG': [-122.3321, -122.3372, -122.3415, -122.3458, -122.3485, -122.3515]
# }

# df = pd.DataFrame(data)

# # Filter rows where INTERV_INIT == "ANC"
# df_anc = df[df['INTERV_INIT'] == 'ANC'].reset_index(drop=True)

# # Initialize a list to store distances
# distances = [0]  # First row has no previous point to compare to, so distance is 0

# # Calculate geodesic distance between consecutive rows
# for i in range(1, len(df_anc)):
#     # Get the coordinates for the current and previous row
#     coords_1 = (df_anc.loc[i - 1, 'ELVIS_USER_LOC2_LAT'], df_anc.loc[i - 1, 'ELVIS_USER_LOC2_LONG'])
#     coords_2 = (df_anc.loc[i, 'ELVIS_USER_LOC2_LAT'], df_anc.loc[i, 'ELVIS_USER_LOC2_LONG'])
    
#     # Calculate distance using geopy
#     distance = geodesic(coords_1, coords_2).miles
#     distances.append(distance)

# # Add the distances to the filtered DataFrame
# df_anc['distance_to_next'] = distances



In [41]:
# df_anc

In [42]:
# Initialize the new 'Distances' column with default value -1
df['Distances'] = -1

# Get unique INTERV_INIT values
unique_interv_init = df['INTERV_INIT'].unique()
unique_interv_init

array(['ANC', 'LSS', 'KSB', 'CBB', 'MAO', 'MXG', 'JMV', 'DHC', 'LXF',
       'NSH', 'CLB', 'GXG', 'TBT', 'SJG', 'JLD', 'LXO', 'BAM', 'DRM',
       'HJT', 'MAD', 'RLC', 'EJP', 'RAB', 'LDP', 'JQT', 'LAJ', 'DKG',
       'PLD', 'DAF', 'BBM', 'DMD', 'HSD', 'PCS', 'RJG', 'CAB', 'RWZ',
       'JLS', 'JTC', 'Pld'], dtype=object)

In [43]:
def get_distance_between_coordinates(lat1, lon1, lat2, lon2):
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
        
        coords_1 = (lat1, lon1)
        coords_2 = (lat2, lon2)
        
        distance = geodesic(coords_1, coords_2).miles
        return distance
    except (ValueError, TypeError) as e:
        # Handle the exception here
        return -1

In [44]:

# Loop through each unique INTERV_INIT value
for interv in unique_interv_init:
    # Filter rows where INTERV_INIT matches the current unique value
    df_filtered = df[df['INTERV_INIT'] == interv]
    df_filtered.sort_values(by='id',ascending=True,inplace=True)
    
#     df_filtered[['ELVIS_USER_LOC2_LAT', 'ELVIS_USER_LOC2_LONG']] = df_filtered[['ELVIS_USER_LOC2_LAT', 'ELVIS_USER_LOC2_LONG']].fillna(method='bfill').fillna(method='ffill')
#     df_filtered[['ELVIS_USER_LOC2_LAT', 'ELVIS_USER_LOC2_LONG']] = df_filtered[['ELVIS_USER_LOC2_LAT', 'ELVIS_USER_LOC2_LONG']].fillna(method='ffill')


    # If only one row is present, keep the distance as -1
    if len(df_filtered) == 1:
        continue
    
    # Calculate the geodesic distance between consecutive rows
    distances = [0]  # Start with 0 for the first row

    for i in range(1, len(df_filtered)):
        # Get the coordinates for the current and previous row
        coords_1 = (df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LONG'])
        coords_2 = (df_filtered.iloc[i]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i]['ELVIS_USER_LOC_LONG'])
        # Calculate distance using geopy
        distance = get_distance_between_coordinates(df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LAT'], df_filtered.iloc[i - 1]['ELVIS_USER_LOC_LONG'],df_filtered.iloc[i]['ELVIS_USER_LOC_LAT'],df_filtered.iloc[i]['ELVIS_USER_LOC_LONG'])
        distances.append(round(distance,4))
    
    # Update the distances in the original DataFrame
    df.loc[df['INTERV_INIT'] == interv, 'Distances'] = distances


In [45]:
round(0.706602,3)

0.707

In [46]:
df[(df['INTERV_INIT'] == 'LSS')][['id','INTERV_INIT','ELVIS_USER_LOC2_LAT', 'ELVIS_USER_LOC2_LONG','Distances']]

,id,INTERV_INIT,ELVIS_USER_LOC2_LAT,ELVIS_USER_LOC2_LONG,Distances
20,11670,LSS,40.163933,-105.096118,0.0000
24,11686,LSS,40.172475,-105.108706,0.7066
25,11695,LSS,40.163830,-105.105010,0.6953
26,11702,LSS,40.137704,-105.130016,2.2365
28,11704,LSS,40.151095,-105.114832,0.9191
...,...,...,...,...,...
3564,17703,LSS,40.172458,-105.108659,0.2324
3570,17710,LSS,40.188243,-105.088457,1.0922
3626,17784,LSS,40.173745,-105.104821,1.0963
3723,17920,LSS,40.139032,-105.121267,3.3261


In [47]:
reviewer_distance_df=df[(df['INTERV_INIT'] == 'ANC')][['id','INTERV_INIT','ELVIS_USER_LOC_LAT', 'ELVIS_USER_LOC_LONG','Distances']]

In [48]:
reviewer_distance_df[['Distances']]

,Distances
4,0.0000
5,-1.0000
6,-1.0000
7,-1.0000
8,0.0000
...,...
3544,2.2245
3556,0.0018
3558,0.0040
3572,1.0832


In [49]:
flags = []

# Loop through the Distances column in chunks of 4
for i in range(0, len(reviewer_distance_df), 3):
    # Extract the chunk of 4 consecutive values
    chunk = reviewer_distance_df['Distances'].iloc[i:i+3]
    
    # Check if all 4 values are the same
    if chunk.nunique() == 1:
        flags.extend([1]*4)  # Flag all 4 as 1
    else:
        flags.extend([0]*4)  # Flag all 4 as 0

# If the length of flags exceeds the DataFrame length (in case of non-multiples of 4)
flags = flags[:len(reviewer_distance_df)]

# Assign the flags to a new column
reviewer_distance_df['Consecutive_Flag'] = flags

In [50]:
reviewer_distance_df[reviewer_distance_df['Consecutive_Flag']==1]

,id,INTERV_INIT,ELVIS_USER_LOC_LAT,ELVIS_USER_LOC_LONG,Distances,Consecutive_Flag


In [51]:
# reviewer_distance_df.to_csv('Interviewer_Survey_Distances.csv',index=False)